In [ ]:

"""
CN7 06 - PR-AUC / Recall / Precision 중심 Feature Engineering Screening
======================================================================

목표
----
CN7_05 결과:
- Repeated Group CV에서는 PR-AUC 약 0.5
- Purged Time-Block CV에서는 PR-AUC 약 0.03~0.04

즉 단순 센서 절대값은 특정 생산구간/상태를 학습하고 있을 가능성이 있다.

이번 단계에서는:
1) raw sensors
2) EDA 상위 변수 축소
3) 공정 관계형 파생변수
4) raw + 관계형 파생변수
5) Side 포함/미포함

을 동일한 CV에서 비교한다.

중요
----
- TimeStamp / 날짜 / PART_FACT_SERIAL / Reason은 feature로 사용하지 않는다.
- rolling / lag / 미래값은 사용하지 않는다.
- 동일 sensor vector group은 반드시 같은 fold에 둔다.
- Primary metric: PR-AUC
- Operational metrics: Recall>=80/90%에서 Precision
"""

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 경로 / 설정
# =============================================================================

CN7_WORK_DIR = Path(
    r"../../outputs/research"
)

RAW_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

OUT_DIR = CN7_WORK_DIR / "cn7_06_feature_engineering"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "PassOrFail"

RANDOM_SEEDS = [42, 52, 62, 72, 82]
N_SPLITS = 5

TIME_BLOCK_MINUTES = 10
PURGE_MINUTES = 10

N_TREES = 500

# CN7_03 단변량 상위권
TOP10_RAW = [
    "Max_Injection_Speed",
    "Average_Back_Pressure",
    "Plasticizing_Position",
    "Max_Back_Pressure",
    "Mold_Temperature_4",
    "Max_Injection_Pressure",
    "Clamp_Close_Time",
    "Mold_Temperature_3",
    "Filling_Time",
    "Max_Switch_Over_Pressure",
]

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 115)
    print(title)
    print("=" * 115)


def safe_ratio(a, b, eps=1e-9):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")

    out = a / b.where(b.abs() > eps, np.nan)

    return out.replace(
        [np.inf, -np.inf],
        np.nan,
    )


def add_engineered_features(df):
    """
    한 행 내부 변수만 사용.
    시간/이웃 행을 전혀 보지 않는다.
    """
    x = df.copy()

    # -------------------------------------------------------------------------
    # 시간 관계
    # -------------------------------------------------------------------------
    x["FE_Injection_minus_Filling_Time"] = (
        x["Injection_Time"]
        - x["Filling_Time"]
    )

    x["FE_Injection_to_Filling_Ratio"] = safe_ratio(
        x["Injection_Time"],
        x["Filling_Time"],
    )

    x["FE_Plasticizing_to_Cycle_Ratio"] = safe_ratio(
        x["Plasticizing_Time"],
        x["Cycle_Time"],
    )

    x["FE_Filling_to_Cycle_Ratio"] = safe_ratio(
        x["Filling_Time"],
        x["Cycle_Time"],
    )

    x["FE_Injection_to_Cycle_Ratio"] = safe_ratio(
        x["Injection_Time"],
        x["Cycle_Time"],
    )

    # -------------------------------------------------------------------------
    # 사출 속도 x 시간
    # -------------------------------------------------------------------------
    x["FE_InjSpeed_x_FillingTime"] = (
        x["Max_Injection_Speed"]
        * x["Filling_Time"]
    )

    x["FE_InjSpeed_x_InjectionTime"] = (
        x["Max_Injection_Speed"]
        * x["Injection_Time"]
    )

    x["FE_InjSpeed_div_FillingTime"] = safe_ratio(
        x["Max_Injection_Speed"],
        x["Filling_Time"],
    )

    # -------------------------------------------------------------------------
    # 압력 관계
    # -------------------------------------------------------------------------
    x["FE_InjectionPressure_minus_SwitchPressure"] = (
        x["Max_Injection_Pressure"]
        - x["Max_Switch_Over_Pressure"]
    )

    x["FE_SwitchPressure_to_InjectionPressure"] = safe_ratio(
        x["Max_Switch_Over_Pressure"],
        x["Max_Injection_Pressure"],
    )

    x["FE_BackPressure_Diff"] = (
        x["Average_Back_Pressure"]
        - x["Max_Back_Pressure"]
    )

    x["FE_BackPressure_Ratio"] = safe_ratio(
        x["Max_Back_Pressure"],
        x["Average_Back_Pressure"],
    )

    x["FE_InjectionPressure_to_BackPressure"] = safe_ratio(
        x["Max_Injection_Pressure"],
        x["Average_Back_Pressure"],
    )

    # -------------------------------------------------------------------------
    # 금형 온도
    # -------------------------------------------------------------------------
    x["FE_MoldTemp_Mean"] = (
        x["Mold_Temperature_3"]
        + x["Mold_Temperature_4"]
    ) / 2

    x["FE_MoldTemp_Diff"] = (
        x["Mold_Temperature_4"]
        - x["Mold_Temperature_3"]
    )

    x["FE_MoldTemp_Ratio"] = safe_ratio(
        x["Mold_Temperature_4"],
        x["Mold_Temperature_3"],
    )

    # -------------------------------------------------------------------------
    # 배럴 온도 profile
    # -------------------------------------------------------------------------
    barrel_cols = [
        "Barrel_Temperature_1",
        "Barrel_Temperature_2",
        "Barrel_Temperature_3",
        "Barrel_Temperature_4",
        "Barrel_Temperature_5",
        "Barrel_Temperature_6",
    ]

    x["FE_BarrelTemp_Mean"] = (
        x[barrel_cols].mean(axis=1)
    )

    x["FE_BarrelTemp_Std"] = (
        x[barrel_cols].std(axis=1, ddof=0)
    )

    x["FE_BarrelTemp_Range"] = (
        x[barrel_cols].max(axis=1)
        - x[barrel_cols].min(axis=1)
    )

    x["FE_BarrelTemp_1_minus_6"] = (
        x["Barrel_Temperature_1"]
        - x["Barrel_Temperature_6"]
    )

    for i in range(1, 6):
        x[
            f"FE_BarrelTemp_{i}_minus_{i+1}"
        ] = (
            x[f"Barrel_Temperature_{i}"]
            - x[f"Barrel_Temperature_{i+1}"]
        )

    x["FE_Hopper_minus_Barrel6"] = (
        x["Hopper_Temperature"]
        - x["Barrel_Temperature_6"]
    )

    # -------------------------------------------------------------------------
    # 스크류 / 위치 관계
    # -------------------------------------------------------------------------
    x["FE_ScrewRPM_Ratio"] = safe_ratio(
        x["Max_Screw_RPM"],
        x["Average_Screw_RPM"],
    )

    x["FE_PlasticizingPosition_minus_Cushion"] = (
        x["Plasticizing_Position"]
        - x["Cushion_Position"]
    )

    x["FE_PlasticizingPosition_to_Cushion"] = safe_ratio(
        x["Plasticizing_Position"],
        x["Cushion_Position"],
    )

    # -------------------------------------------------------------------------
    # 핵심 EDA feature 간 상호작용
    # -------------------------------------------------------------------------
    x["FE_Speed_x_PlasticizingPosition"] = (
        x["Max_Injection_Speed"]
        * x["Plasticizing_Position"]
    )

    x["FE_Speed_x_MoldTempMean"] = (
        x["Max_Injection_Speed"]
        * x["FE_MoldTemp_Mean"]
    )

    x["FE_BackPressure_x_MoldTempMean"] = (
        x["Average_Back_Pressure"]
        * x["FE_MoldTemp_Mean"]
    )

    return x


def make_model(model_name, seed):
    if model_name == "logreg_standard":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    C=0.1,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=seed,
                ),
            ),
        ])

    if model_name == "logreg_robust":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                ),
            ),
            (
                "scaler",
                RobustScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    C=0.1,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=seed,
                ),
            ),
        ])

    if model_name == "extratrees":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                ),
            ),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=N_TREES,
                    max_depth=None,
                    min_samples_leaf=2,
                    max_features="sqrt",
                    class_weight="balanced",
                    random_state=seed,
                    n_jobs=-1,
                ),
            ),
        ])

    raise ValueError(model_name)


def score_predictions(y_true, score):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    out = {
        "n": len(y),
        "defects": int(y.sum()),
    }

    if len(y) == 0 or len(np.unique(y)) < 2:
        out["PR_AUC"] = np.nan
        out["ROC_AUC"] = np.nan
        return out

    out["PR_AUC"] = average_precision_score(
        y,
        s,
    )

    out["ROC_AUC"] = roc_auc_score(
        y,
        s,
    )

    return out


def threshold_table(y_true, score):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    thresholds = np.sort(
        np.unique(s)
    )[::-1]

    rows = []

    for t in thresholds:
        pred = (s >= t).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y,
            pred,
            labels=[0, 1],
        ).ravel()

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        rows.append({
            "threshold": float(t),
            "precision": precision,
            "recall": recall,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "predicted_positive": int(tp + fp),
            "predicted_positive_rate": (
                (tp + fp) / len(y)
            ),
        })

    return pd.DataFrame(rows)


def operating_metric(y, score, recall_target):
    tab = threshold_table(
        y,
        score,
    )

    cand = tab[
        tab["recall"] >= recall_target
    ].copy()

    if len(cand) == 0:
        return {
            "threshold": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "TP": np.nan,
            "FP": np.nan,
            "FN": np.nan,
            "predicted_positive_rate": np.nan,
        }

    cand = cand.sort_values(
        [
            "precision",
            "recall",
            "predicted_positive",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    r = cand.iloc[0]

    return {
        "threshold": float(r["threshold"]),
        "precision": float(r["precision"]),
        "recall": float(r["recall"]),
        "TP": int(r["TP"]),
        "FP": int(r["FP"]),
        "FN": int(r["FN"]),
        "predicted_positive_rate": float(
            r["predicted_positive_rate"]
        ),
    }


def top_fraction(y_true, score, frac):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    k = max(
        1,
        int(np.ceil(len(y) * frac)),
    )

    order = np.argsort(-s)
    idx = order[:k]

    tp = int(y[idx].sum())
    fp = int(k - tp)
    fn = int(y.sum() - tp)

    return {
        "k": k,
        "precision": tp / k,
        "recall": (
            tp / y.sum()
            if y.sum() > 0
            else np.nan
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
    }


def predict_model(
    X,
    y,
    train_idx,
    valid_idx,
    model_name,
    seed,
):
    model = make_model(
        model_name,
        seed,
    )

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
    )

    return model.predict_proba(
        X.iloc[valid_idx]
    )[:, 1]


# =============================================================================
# 1. 데이터
# =============================================================================

banner("CN7 06 - Feature Engineering Screening")

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"없음: {RAW_PATH}"
    )

df = pd.read_csv(RAW_PATH)

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="raise",
).astype(int)

if "TimeStamp_dt" in df.columns:
    df["TimeStamp_dt"] = pd.to_datetime(
        df["TimeStamp_dt"],
        errors="coerce",
    )
else:
    df["TimeStamp_dt"] = pd.to_datetime(
        df["TimeStamp"],
        errors="coerce",
    )

if "Side" not in df.columns:
    s = df["PART_NAME"].astype(str).str.upper()

    df["Side"] = np.where(
        s.str.contains(
            r"\bRH\b",
            regex=True,
            na=False,
        ),
        "RH",
        "LH",
    )

df["Side_RH"] = (
    df["Side"] == "RH"
).astype(int)

print(
    f"shape={df.shape}, "
    f"defects={df[TARGET].sum()} / {len(df)}"
)


# =============================================================================
# 2. 상수 제거 + 파생변수
# =============================================================================

banner("1) Feature 생성")

constant_cols = [
    c for c in SENSOR_COLS
    if c in df.columns
    and df[c].nunique(dropna=False) <= 1
]

base_sensors = [
    c for c in SENSOR_COLS
    if c in df.columns
    and c not in constant_cols
]

fe_df = add_engineered_features(
    df
)

engineered_cols = [
    c for c in fe_df.columns
    if c.startswith("FE_")
]

print("base sensors:", len(base_sensors))
print("constant removed:", constant_cols)
print("engineered:", len(engineered_cols))

for c in engineered_cols:
    print(" ", c)

# EDA top10에 상수/누락 없도록
top10 = [
    c for c in TOP10_RAW
    if c in base_sensors
]

FEATURE_SETS = {
    "base": base_sensors,

    "base_side":
        base_sensors
        + ["Side_RH"],

    "top10_side":
        top10
        + ["Side_RH"],

    "engineered_only_side":
        engineered_cols
        + ["Side_RH"],

    "base_engineered_side":
        base_sensors
        + engineered_cols
        + ["Side_RH"],

    "top10_engineered_side":
        list(dict.fromkeys(
            top10
            + engineered_cols
            + ["Side_RH"]
        )),
}

for name, cols in FEATURE_SETS.items():
    print(
        f"{name:28s} : "
        f"{len(cols)} features"
    )


# =============================================================================
# 3. Sensor Group
# =============================================================================

banner("2) Group 생성")

df["sensor_group_id"] = (
    df.groupby(
        base_sensors,
        dropna=False,
        sort=False,
    )
    .ngroup()
)

groups = df["sensor_group_id"]
y = df[TARGET]

print(
    "groups:",
    groups.nunique(),
)

positive_groups = (
    df.groupby(
        "sensor_group_id"
    )[TARGET]
    .max()
    .sum()
)

print(
    "positive groups:",
    int(positive_groups),
)


# =============================================================================
# 4. Repeated StratifiedGroup CV
# =============================================================================

banner("3) Repeated StratifiedGroup CV")

MODEL_NAMES = [
    "logreg_standard",
    "logreg_robust",
    "extratrees",
]

group_results = []
group_oof_rows = []

for feature_name, cols in FEATURE_SETS.items():

    X = fe_df[cols].astype(float)

    for model_name in MODEL_NAMES:

        pred_sum = np.zeros(
            len(df),
            dtype=float,
        )

        pred_count = np.zeros(
            len(df),
            dtype=int,
        )

        fold_pr = []

        for repeat, seed in enumerate(
            RANDOM_SEEDS,
            start=1,
        ):
            cv = StratifiedGroupKFold(
                n_splits=N_SPLITS,
                shuffle=True,
                random_state=seed,
            )

            for fold, (
                tr_idx,
                va_idx,
            ) in enumerate(
                cv.split(
                    X,
                    y,
                    groups,
                ),
                start=1,
            ):
                pred = predict_model(
                    X=X,
                    y=y,
                    train_idx=tr_idx,
                    valid_idx=va_idx,
                    model_name=model_name,
                    seed=seed + fold,
                )

                pred_sum[va_idx] += pred
                pred_count[va_idx] += 1

                if y.iloc[va_idx].nunique() == 2:
                    fold_pr.append(
                        average_precision_score(
                            y.iloc[va_idx],
                            pred,
                        )
                    )

        oof = np.divide(
            pred_sum,
            pred_count,
            out=np.full(
                len(df),
                np.nan,
            ),
            where=pred_count > 0,
        )

        metric = score_predictions(
            y,
            oof,
        )

        op80 = operating_metric(
            y,
            oof,
            0.80,
        )

        op90 = operating_metric(
            y,
            oof,
            0.90,
        )

        top05 = top_fraction(
            y,
            oof,
            0.05,
        )

        result = {
            "cv": "repeated_group",
            "feature_set": feature_name,
            "model": model_name,
            "n_features": len(cols),
            "PR_AUC": metric["PR_AUC"],
            "ROC_AUC_ref": metric["ROC_AUC"],

            "R80_precision": op80["precision"],
            "R80_recall": op80["recall"],
            "R80_threshold": op80["threshold"],
            "R80_TP": op80["TP"],
            "R80_FP": op80["FP"],

            "R90_precision": op90["precision"],
            "R90_recall": op90["recall"],
            "R90_threshold": op90["threshold"],
            "R90_TP": op90["TP"],
            "R90_FP": op90["FP"],

            "Top5_precision": top05["precision"],
            "Top5_recall": top05["recall"],
            "Top5_TP": top05["TP"],
            "Top5_FP": top05["FP"],

            "fold_PR_mean": (
                np.mean(fold_pr)
                if fold_pr
                else np.nan
            ),
            "fold_PR_std": (
                np.std(
                    fold_pr,
                    ddof=1,
                )
                if len(fold_pr) > 1
                else np.nan
            ),
        }

        group_results.append(
            result
        )

        for i in range(len(df)):
            group_oof_rows.append({
                "row_index": i,
                "feature_set": feature_name,
                "model": model_name,
                "target": int(y.iloc[i]),
                "prediction": oof[i],
                "sensor_group_id": int(
                    groups.iloc[i]
                ),
            })

        print(
            f"{feature_name:28s} | "
            f"{model_name:16s} | "
            f"PR={metric['PR_AUC']:.4f} | "
            f"R80 P={op80['precision']:.3f} "
            f"R={op80['recall']:.3f} | "
            f"R90 P={op90['precision']:.3f} "
            f"R={op90['recall']:.3f}"
        )


# =============================================================================
# 5. Purged Time-Block CV
# =============================================================================

banner("4) Purged Time-Block CV")

df["time_block"] = (
    df["TimeStamp_dt"]
    .dt.floor(
        f"{TIME_BLOCK_MINUTES}min"
    )
)

unique_blocks = sorted(
    df["time_block"]
    .dropna()
    .unique()
)

block_results = []
block_oof_rows = []

for feature_name, cols in FEATURE_SETS.items():

    X = fe_df[cols].astype(float)

    for model_name in MODEL_NAMES:

        oof = np.full(
            len(df),
            np.nan,
            dtype=float,
        )

        for fold, block in enumerate(
            unique_blocks,
            start=1,
        ):
            block = pd.Timestamp(block)

            val_mask = (
                df["time_block"] == block
            )

            val_idx = np.where(
                val_mask.to_numpy()
            )[0]

            if len(val_idx) == 0:
                continue

            block_start = block
            block_end = (
                block
                + pd.Timedelta(
                    minutes=TIME_BLOCK_MINUTES
                )
            )

            purge_start = (
                block_start
                - pd.Timedelta(
                    minutes=PURGE_MINUTES
                )
            )

            purge_end = (
                block_end
                + pd.Timedelta(
                    minutes=PURGE_MINUTES
                )
            )

            train_mask = ~(
                (
                    df["TimeStamp_dt"]
                    >= purge_start
                )
                &
                (
                    df["TimeStamp_dt"]
                    < purge_end
                )
            )

            val_groups = set(
                df.loc[
                    val_mask,
                    "sensor_group_id",
                ]
            )

            train_mask &= ~df[
                "sensor_group_id"
            ].isin(
                val_groups
            )

            tr_idx = np.where(
                train_mask.to_numpy()
            )[0]

            if (
                len(tr_idx) == 0
                or y.iloc[tr_idx].nunique() < 2
            ):
                continue

            pred = predict_model(
                X=X,
                y=y,
                train_idx=tr_idx,
                valid_idx=val_idx,
                model_name=model_name,
                seed=42 + fold,
            )

            oof[val_idx] = pred

        metric = score_predictions(
            y,
            oof,
        )

        op80 = operating_metric(
            y,
            oof,
            0.80,
        )

        op90 = operating_metric(
            y,
            oof,
            0.90,
        )

        top05 = top_fraction(
            y,
            oof,
            0.05,
        )

        result = {
            "cv": "purged_time_block",
            "feature_set": feature_name,
            "model": model_name,
            "n_features": len(cols),
            "PR_AUC": metric["PR_AUC"],
            "ROC_AUC_ref": metric["ROC_AUC"],

            "R80_precision": op80["precision"],
            "R80_recall": op80["recall"],
            "R80_threshold": op80["threshold"],
            "R80_TP": op80["TP"],
            "R80_FP": op80["FP"],

            "R90_precision": op90["precision"],
            "R90_recall": op90["recall"],
            "R90_threshold": op90["threshold"],
            "R90_TP": op90["TP"],
            "R90_FP": op90["FP"],

            "Top5_precision": top05["precision"],
            "Top5_recall": top05["recall"],
            "Top5_TP": top05["TP"],
            "Top5_FP": top05["FP"],
        }

        block_results.append(
            result
        )

        for i in range(len(df)):
            block_oof_rows.append({
                "row_index": i,
                "feature_set": feature_name,
                "model": model_name,
                "target": int(y.iloc[i]),
                "prediction": oof[i],
                "sensor_group_id": int(
                    groups.iloc[i]
                ),
            })

        print(
            f"{feature_name:28s} | "
            f"{model_name:16s} | "
            f"PR={metric['PR_AUC']:.4f} | "
            f"R80 P={op80['precision']:.3f} "
            f"R={op80['recall']:.3f} | "
            f"R90 P={op90['precision']:.3f} "
            f"R={op90['recall']:.3f}"
        )


# =============================================================================
# 6. 결과 저장
# =============================================================================

banner("5) 결과 요약")

group_df = pd.DataFrame(
    group_results
)

block_df = pd.DataFrame(
    block_results
)

result_df = pd.concat(
    [
        group_df,
        block_df,
    ],
    ignore_index=True,
)

result_df.to_csv(
    OUT_DIR / "feature_engineering_results.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(
    group_oof_rows
).to_csv(
    OUT_DIR / "group_oof_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(
    block_oof_rows
).to_csv(
    OUT_DIR / "purged_oof_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

# feature 목록
feature_rows = []

for name, cols in FEATURE_SETS.items():
    for c in cols:
        feature_rows.append({
            "feature_set": name,
            "feature": c,
            "is_engineered": c.startswith(
                "FE_"
            ),
        })

pd.DataFrame(
    feature_rows
).to_csv(
    OUT_DIR / "feature_sets.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 7. Group 성능표
# =============================================================================

banner("6) Repeated Group CV 순위")

g = group_df.sort_values(
    [
        "PR_AUC",
        "R80_precision",
    ],
    ascending=[
        False,
        False,
    ],
)

print(
    g[
        [
            "feature_set",
            "model",
            "n_features",
            "PR_AUC",
            "R80_precision",
            "R80_recall",
            "R90_precision",
            "R90_recall",
            "Top5_precision",
            "Top5_recall",
            "fold_PR_mean",
            "fold_PR_std",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 8. Purged 성능표
# =============================================================================

banner("7) Purged Time-Block CV 순위")

b = block_df.sort_values(
    [
        "PR_AUC",
        "R80_precision",
    ],
    ascending=[
        False,
        False,
    ],
)

print(
    b[
        [
            "feature_set",
            "model",
            "n_features",
            "PR_AUC",
            "R80_precision",
            "R80_recall",
            "R90_precision",
            "R90_recall",
            "Top5_precision",
            "Top5_recall",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 9. Robustness score
# =============================================================================

banner("8) Group + Purged 동시 비교")

merged = group_df.merge(
    block_df,
    on=[
        "feature_set",
        "model",
        "n_features",
    ],
    suffixes=(
        "_group",
        "_purged",
    ),
)

# 단순 보조 지표:
# 낮은 쪽 PR-AUC를 중시
merged["PR_min"] = merged[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].min(axis=1)

merged["PR_mean"] = merged[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].mean(axis=1)

merged["R80_precision_min"] = merged[
    [
        "R80_precision_group",
        "R80_precision_purged",
    ]
].min(axis=1)

merged = merged.sort_values(
    [
        "PR_min",
        "PR_mean",
        "R80_precision_min",
    ],
    ascending=False,
)

print(
    merged[
        [
            "feature_set",
            "model",
            "n_features",
            "PR_AUC_group",
            "PR_AUC_purged",
            "PR_min",
            "R80_precision_group",
            "R80_precision_purged",
            "R80_recall_group",
            "R80_recall_purged",
            "R90_precision_group",
            "R90_precision_purged",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

merged.to_csv(
    OUT_DIR / "group_vs_purged_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 10. 기준 모델 대비 변화
# =============================================================================

banner("9) 기존 baseline 대비 변화")

# 기존 핵심 baseline:
# base_side + logreg_standard
baseline_key = (
    (merged["feature_set"] == "base_side")
    &
    (merged["model"] == "logreg_standard")
)

if baseline_key.any():

    base = merged.loc[
        baseline_key
    ].iloc[0]

    delta = merged[
        [
            "feature_set",
            "model",
            "n_features",
            "PR_AUC_group",
            "PR_AUC_purged",
            "R80_precision_group",
            "R80_precision_purged",
        ]
    ].copy()

    delta["delta_PR_group"] = (
        delta["PR_AUC_group"]
        - base["PR_AUC_group"]
    )

    delta["delta_PR_purged"] = (
        delta["PR_AUC_purged"]
        - base["PR_AUC_purged"]
    )

    delta["delta_R80P_group"] = (
        delta["R80_precision_group"]
        - base["R80_precision_group"]
    )

    delta["delta_R80P_purged"] = (
        delta["R80_precision_purged"]
        - base["R80_precision_purged"]
    )

    delta = delta.sort_values(
        [
            "delta_PR_purged",
            "delta_PR_group",
        ],
        ascending=False,
    )

    print(
        delta.round(4).to_string(
            index=False
        )
    )

    delta.to_csv(
        OUT_DIR / "baseline_delta.csv",
        index=False,
        encoding="utf-8-sig",
    )


# =============================================================================
# 11. 자동 요약
# =============================================================================

banner("10) 자동 요약")

lines = []

best_group = g.iloc[0]
best_purged = b.iloc[0]
best_robust = merged.iloc[0]

lines.append(
    f"- Group PR-AUC 최고: "
    f"{best_group['feature_set']} / "
    f"{best_group['model']} | "
    f"PR={best_group['PR_AUC']:.4f}, "
    f"R80 Precision={best_group['R80_precision']:.4f}, "
    f"Recall={best_group['R80_recall']:.4f}"
)

lines.append(
    f"- Purged PR-AUC 최고: "
    f"{best_purged['feature_set']} / "
    f"{best_purged['model']} | "
    f"PR={best_purged['PR_AUC']:.4f}, "
    f"R80 Precision={best_purged['R80_precision']:.4f}, "
    f"Recall={best_purged['R80_recall']:.4f}"
)

lines.append(
    f"- 두 CV의 낮은 PR-AUC(PR_min)가 가장 높은 조합: "
    f"{best_robust['feature_set']} / "
    f"{best_robust['model']} | "
    f"Group={best_robust['PR_AUC_group']:.4f}, "
    f"Purged={best_robust['PR_AUC_purged']:.4f}"
)

lines.append(
    "- 해석 원칙: Group 성능만 크게 오르고 Purged가 오르지 않으면 "
    "해당 feature는 생산구간 특이 패턴을 학습했을 가능성을 의심한다."
)

lines.append(
    "- 다음 단계에서는 Group/Purged 모두에서 개선된 feature set만 남겨 "
    "하이퍼파라미터/앙상블을 실험한다."
)

for line in lines:
    print(line)

with open(
    OUT_DIR / "feature_engineering_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(lines))


# =============================================================================
# 완료
# =============================================================================

banner("완료")

print("결과 폴더:", OUT_DIR)
print()
print("핵심 파일:")
print("  1) feature_engineering_results.csv")
print("  2) group_vs_purged_comparison.csv")
print("  3) baseline_delta.csv")
print("  4) feature_sets.csv")
print("  5) group_oof_predictions.csv")
print("  6) purged_oof_predictions.csv")
print("  7) feature_engineering_summary.txt")
print()
print(
    "실행 후 6) Repeated Group CV 순위부터 "
    "10) 자동 요약까지 보내주세요."
)
